# 03 — Tissue Segmentation

Use the skull-stripped brain and mask from notebook 02, then run the pretrained **deepmriprep** tissue-segmentation submodels to estimate **gray matter (GM)**, **white matter (WM)**, and **cerebrospinal fluid (CSF)**.

Why this model?
- It is PyTorch-based and directly outputs GM / WM / CSF probability maps.
- We can call its internal segmentation steps directly, so we **reuse** the skull-stripped outputs from notebook 02 instead of restarting from raw data.
- It performs an internal affine registration to template space before tissue inference, then maps the results back to the native scan space.

**Outputs saved by this notebook**
- `outputs/segmentations/gm_probability.nii.gz`
- `outputs/segmentations/wm_probability.nii.gz`
- `outputs/segmentations/csf_probability.nii.gz`
- `outputs/segmentations/tissue_segmentation_3class.nii.gz`
- `outputs/segmentations/tissue_volumes.csv`

**Runtime note:** on the first run, `deepmriprep` will download its pretrained weights automatically. On Apple Silicon, run it in CPU mode.

In [ ]:
import warnings

import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path

warnings.filterwarnings("ignore", message=r"torch\.meshgrid:.*")
warnings.filterwarnings(
    "ignore",
    message=r"Using a non-tuple sequence for multidimensional indexing is deprecated.*",
)

from deepmriprep import Preprocess

In [ ]:
# ── Paths ─────────────────────────────────
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "data").exists() and (CWD / "outputs").exists() else CWD.parent
assert (ROOT / "data").exists(), f"Could not locate project root from {CWD}"
PROC = ROOT / "data" / "processed"
FIGURES = ROOT / "outputs" / "figures"
SEG_DIR = ROOT / "outputs" / "segmentations"

FIGURES.mkdir(parents=True, exist_ok=True)
SEG_DIR.mkdir(parents=True, exist_ok=True)

STRIPPED = PROC / "brain_stripped.nii.gz"
MASK = PROC / "brain_stripped_mask.nii.gz"

AFFINE_BRAIN = PROC / "brain_affine_template_space.nii.gz"
AFFINE_MASK = PROC / "brain_mask_template_space.nii.gz"
COARSE_TISSUE = SEG_DIR / "tissue_segmentation_coarse.nii.gz"
GM_PROB = SEG_DIR / "gm_probability.nii.gz"
WM_PROB = SEG_DIR / "wm_probability.nii.gz"
CSF_PROB = SEG_DIR / "csf_probability.nii.gz"
HARD_SEG = SEG_DIR / "tissue_segmentation_3class.nii.gz"
VOLUMES_CSV = SEG_DIR / "tissue_volumes.csv"

In [ ]:
# ── Reusable helpers ──────────────────────────
SEG_CMAP = ListedColormap(["#d95f02", "#1b9e77", "#7570b3"])


def get_data(img):
    return np.asarray(img.get_fdata(), dtype=np.float32)


def save_nifti(reference_img, array, path, dtype=np.float32):
    header = reference_img.header.copy()
    header.set_data_dtype(np.dtype(dtype))
    nib.save(nib.Nifti1Image(array.astype(dtype), reference_img.affine, header), str(path))


def plot_middle_slices(volume, title="Middle Slices", cmap="gray", save_path=None, vmin=None, vmax=None):
    x_mid, y_mid, z_mid = [s // 2 for s in volume.shape]
    slices = {
        "Sagittal": np.rot90(volume[x_mid, :, :]),
        "Coronal": np.rot90(volume[:, y_mid, :]),
        "Axial": np.rot90(volume[:, :, z_mid]),
    }

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (name, slc) in zip(axes, slices.items()):
        ax.imshow(slc, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(name)
        ax.axis("off")

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_probability_maps(image, gm, wm, csf, save_path=None):
    z_mid = image.shape[2] // 2
    panels = [
        ("Input (Axial)", np.rot90(image[:, :, z_mid]), "gray", None, None),
        ("GM probability", np.rot90(gm[:, :, z_mid]), "magma", 0, 1),
        ("WM probability", np.rot90(wm[:, :, z_mid]), "viridis", 0, 1),
        ("CSF probability", np.rot90(csf[:, :, z_mid]), "cividis", 0, 1),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for ax, (title, slc, cmap, vmin, vmax) in zip(axes, panels):
        im = ax.imshow(slc, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.axis("off")
        if title != "Input (Axial)":
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_segmentation_overlay(image, labels, save_path=None):
    x_mid, y_mid, z_mid = [s // 2 for s in image.shape]
    slices = [
        ("Sagittal", np.rot90(image[x_mid, :, :]), np.rot90(labels[x_mid, :, :])),
        ("Coronal", np.rot90(image[:, y_mid, :]), np.rot90(labels[:, y_mid, :])),
        ("Axial", np.rot90(image[:, :, z_mid]), np.rot90(labels[:, :, z_mid])),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (name, base, seg) in zip(axes, slices):
        ax.imshow(base, cmap="gray")
        masked = np.ma.masked_where(seg == 0, seg)
        ax.imshow(masked, cmap=SEG_CMAP, alpha=0.45, interpolation="none", vmin=1, vmax=3)
        ax.set_title(name)
        ax.axis("off")

    fig.suptitle("GM / WM / CSF Hard Segmentation Overlay", fontsize=14)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

## Step 1 — Load the skull-stripped brain and mask

These are the outputs from notebook 02. We keep the pipeline modular by using them as the starting point for tissue segmentation.

In [ ]:
for path in [STRIPPED, MASK]:
    assert path.exists(), f"Missing required file: {path}"

brain_img = nib.load(str(STRIPPED))
mask_img = nib.load(str(MASK))

brain = get_data(brain_img)
mask = get_data(mask_img) > 0.5

print(f"Brain shape : {brain.shape}")
print(f"Mask shape  : {mask.shape}")
print(f"Voxel size  : {brain_img.header.get_zooms()[:3]}")
print(f"Brain voxels: {mask.sum():,}")

plot_middle_slices(
    brain,
    title="Input to Tissue Segmentation — Skull-Stripped Brain",
    save_path=str(FIGURES / "03_input_brain.png"),
)

## Step 2 — Initialize the pretrained segmenter

We use `deepmriprep` in **CPU mode** because its GPU acceleration targets NVIDIA CUDA. The first run may take a minute or two while model weights download.

In [ ]:
segmenter = Preprocess(no_gpu=True)
print("Runtime device:", segmenter.device)

## Step 3 — Affine registration to template space

The tissue model expects the skull-stripped brain to be aligned to the internal template. We save the affine-aligned brain and mask so this intermediate step is inspectable.

In [ ]:
affine_outputs = segmenter.run_affine_register(brain_img, mask_img)

affine_brain_img = affine_outputs["brain_large"]
affine_mask_img = affine_outputs["mask_large"]
affine_brain = get_data(affine_brain_img)

nib.save(affine_brain_img, str(AFFINE_BRAIN))
nib.save(affine_mask_img, str(AFFINE_MASK))

print(f"Template-space shape : {affine_brain.shape}")
print(f"Affine loss          : {float(affine_outputs['affine_loss'].iloc[0]):.6f}")
print(f"Saved                : {AFFINE_BRAIN.name}")
print(f"Saved                : {AFFINE_MASK.name}")

plot_middle_slices(
    affine_brain,
    title="Step 3 — Affine-Registered Brain in Template Space",
    save_path=str(FIGURES / "03_affine_registered.png"),
)

## Step 4 — Coarse tissue segmentation

`deepmriprep` first predicts a coarse 4-class tissue map in template space, then moves it back to native scan space. This is an internal intermediate representation used to refine the final GM / WM / CSF probabilities.

In [ ]:
coarse_outputs = segmenter.run_segment_brain(
    affine_outputs["brain_large"],
    mask_img,
    affine_outputs["affine"],
    affine_outputs["mask_large"],
)

coarse_native_img = coarse_outputs["p0"]
coarse_native = get_data(coarse_native_img)

save_nifti(brain_img, coarse_native, COARSE_TISSUE, dtype=np.uint8)

print("Unique coarse labels:", np.unique(coarse_native).astype(int))
print("Saved:", COARSE_TISSUE)

plot_middle_slices(
    coarse_native,
    title="Step 4 — Coarse Tissue Segmentation (0=BG, 1=GM, 2=WM, 3=CSF)",
    cmap="viridis",
    vmin=0,
    vmax=3,
    save_path=str(FIGURES / "03_coarse_tissue.png"),
)

## Step 5 — Final GM / WM / CSF probability maps in native space

This stage refines the coarse map into continuous tissue probabilities, then estimates tissue volumes. We also derive a simple hard-label map by taking the highest-probability tissue at each voxel inside the brain mask.

In [ ]:
tissue_outputs = segmenter.run_segment_nogm(
    coarse_outputs["p0_large"],
    affine_outputs["affine"],
    brain_img,
)

gm_prob = get_data(tissue_outputs["p1"]) * mask
wm_prob = get_data(tissue_outputs["p2"]) * mask
csf_prob = get_data(tissue_outputs["p3"]) * mask

stacked = np.stack([gm_prob, wm_prob, csf_prob], axis=0)
hard_labels = np.argmax(stacked, axis=0).astype(np.uint8) + 1
hard_labels[(stacked.sum(axis=0) == 0) | (~mask)] = 0

save_nifti(brain_img, gm_prob, GM_PROB, dtype=np.float32)
save_nifti(brain_img, wm_prob, WM_PROB, dtype=np.float32)
save_nifti(brain_img, csf_prob, CSF_PROB, dtype=np.float32)
save_nifti(brain_img, hard_labels, HARD_SEG, dtype=np.uint8)

volume_table = pd.DataFrame(
    {
        "tissue": ["GM", "WM", "CSF", "TIV"],
        "volume_cm3": [
            float(tissue_outputs["gmv"].iloc[0]),
            float(tissue_outputs["wmv"].iloc[0]),
            float(tissue_outputs["csfv"].iloc[0]),
            float(tissue_outputs["tiv"].iloc[0]),
        ],
        "fraction_of_tiv": [
            float(tissue_outputs["rel_gmv"].iloc[0]),
            float(tissue_outputs["rel_wmv"].iloc[0]),
            float(tissue_outputs["rel_csfv"].iloc[0]),
            1.0,
        ],
    }
)
volume_table.to_csv(VOLUMES_CSV, index=False)

for name, arr in [("GM", gm_prob), ("WM", wm_prob), ("CSF", csf_prob)]:
    print(f"{name:>3} probability range: {arr.min():.4f} → {arr.max():.4f}")

print("\nSaved outputs:")
print(" -", GM_PROB)
print(" -", WM_PROB)
print(" -", CSF_PROB)
print(" -", HARD_SEG)
print(" -", VOLUMES_CSV)

volume_table

In [ ]:
plot_probability_maps(
    brain,
    gm_prob,
    wm_prob,
    csf_prob,
    save_path=str(FIGURES / "03_tissue_probabilities_axial.png"),
)

plot_segmentation_overlay(
    brain,
    hard_labels,
    save_path=str(FIGURES / "03_tissue_overlay.png"),
)

## Step 6 — Summary and sanity checks

At this point you should have:
- three probability volumes (`GM`, `WM`, `CSF`),
- one hard-label segmentation volume (`1=GM`, `2=WM`, `3=CSF`),
- a CSV table with tissue volumes.

If the overlay looks anatomically sensible, the repository is ready for a final notebook focused on packaging results, quality control, and portfolio-ready visual outputs.

In [ ]:
print("=" * 60)
print("  TISSUE SEGMENTATION — SUMMARY")
print("=" * 60)
print(f"Input brain shape        : {brain.shape}")
print(f"Template brain shape     : {affine_brain.shape}")
print(f"Hard-label classes       : {np.unique(hard_labels).astype(int)}")
print(f"GM voxels  (>0.5 prob)   : {(gm_prob > 0.5).sum():,}")
print(f"WM voxels  (>0.5 prob)   : {(wm_prob > 0.5).sum():,}")
print(f"CSF voxels (>0.5 prob)   : {(csf_prob > 0.5).sum():,}")
print(f"Outputs directory        : {SEG_DIR}")

print("\nLegend:")
print("  1 = Gray Matter (GM)")
print("  2 = White Matter (WM)")
print("  3 = Cerebrospinal Fluid (CSF)")

volume_table